In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import crosscoders as xc
import torch, nnsight


------------------------- CONSTANTS -------------------------
GlobalsConfig(
    PROJECT_ROOT_DIR = '/home/ec2-user/crosscoders',
    CONFIG_FILEPATH = '/home/ec2-user/crosscoders/src/scripts/configs/train.yml',
    EXPERIMENT = ExperimentConfig(
        BATCH_SIZE = 8192,
        MAX_RECORDS = None,
        MAX_BATCHES = 1000000,
        MAX_TOKENS = 100000,
        NUM_GPUS = 1,
        NUM_TRAINERS = 1,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
)
-------------------------------------------------------------



In [2]:
from crosscoders.constants import CONSTANTS
from crosscoders.dataclasses.configs.runner import RunnerConfig
from crosscoders.utils import from_dict, get_config


runner_cfg = from_dict(
    RunnerConfig,
    get_config(CONSTANTS.CONFIG_FILEPATH).get('RUNNER', {})
)
runner_cfg

RunnerConfig(
    MODEL = ModelConfig(
        CAUSALITY = 'acausal',
        LOCALITY = 'global',
        N_LAYERS = 12,
        D_MODEL = 768,
        D_CODER = 16384,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
    LOSS = AcausalLossConfig(
        L1_COEFFICIENT = 1.0,
    ),
    OPTIMIZER = OptimizerConfig(
        optimizer = <class 'torch.optim.adam.Adam'>,
        parameters = OptimizerParameters(
            lr = 0.0004,
            betas = (0.9, 0.999),
            fused = True,
        ),
    ),
)

In [24]:
from crosscoders.data.dataset import TinyStoriesRayDataset


train_ds = TinyStoriesRayDataset().load('activations')



train_dl = train_ds.iter_torch_batches(
    batch_size=CONSTANTS.EXPERIMENT.BATCH_SIZE,
    # local_shuffle_buffer_size=16
    device=CONSTANTS.EXPERIMENT.HARDWARE.device
)

Metadata Fetch Progress 0:   0%|          | 0.00/83.0 [00:00<?, ? task/s]

Parquet Files Sample 0:   0%|          | 0.00/5.00 [00:00<?, ? file/s]

In [20]:
from crosscoders.autoencoders.acausal.runner import AcausalAutoencoderRunner


runner = AcausalAutoencoderRunner(runner_cfg)

In [14]:
from nnsight import LanguageModel
from transformer_lens import HookedTransformer

# model = LanguageModel('openai-community/gpt2')
model = HookedTransformer.from_pretrained('gpt2-small')
model.requires_grad_(False)
model

Loaded pretrained model gpt2-small into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

In [16]:
# for name, param in model.named_parameters():
#     print(f"Layer: {name}, Requires gradient: {param.requires_grad}")

In [17]:
import nnsight
from nnsight import NNsight

model = NNsight(model)
model

HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

In [25]:
x = next(iter(train_dl))['resid_post']

x_hat = runner.model(x)

2025-02-12 17:01:58,539	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-12_16-58-48_177960_6332/logs/ray-data
2025-02-12 17:01:58,539	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=100000]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- limit=100000 2: 0.00 row [00:00, ? row/s]

In [ ]:
def acti(x, x_hat, model):

    tokens = model.to_tokens(x.tolist())


    # with torch.inference_mode():
    with model.trace(x):
        logits = model.output.save()

    print(logits)


    with model.trace(x):

        # before = model.blocks[0].output.clone().save()
        
        for i in range(runner_cfg.MODEL.N_LAYERS):
            model.blocks[i].output = x_hat[:,i,:]

        # after = model.blocks[0].output.save()



acti(x, x_hat, model)

In [26]:
x_hat.shape

torch.Size([8192, 12, 768])

In [52]:
model.blocks[0].output

ReferenceError: weakly-referenced object no longer exists

In [53]:
with model.trace(x):

    before = model.blocks[0].output.clone().save()
    
    model.blocks[0].output = x_hat[:,0,:]

    after = model.blocks[0].output.save()


print(before)
print(after)

NNsightError: tensors used as indices must be long, int, byte or bool tensors